# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and analyze the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL and is designed to be FAIR and AI-ready.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review the available record sets, fields, and their `@id`s.

Below we display key metadata components and enumerate record sets for exploration.

In [ ]:
# View top-level metadata
print(f"Dataset title: {metadata.name}")
print(f"Version: {getattr(metadata, 'version', None)}")
print(f"Publication date: {getattr(metadata, 'datePublished', None)}")
print(f"Keywords: {getattr(metadata, 'keywords', None)}\n")

# List record sets and their fields by @id
print("Available record sets and their fields:\n")
all_record_sets = list(dataset.record_sets())
record_set_ids = []

for record_set in all_record_sets:
    print(f"RecordSet name: {record_set.name}")
    print(f"@id: {record_set.id}")
    record_set_ids.append(record_set.id)
    print("Fields (by @id):")
    for field in record_set.fields:
        print(f"  Field name: {getattr(field, 'name', None)}, @id: {field.id}, dataType: {getattr(field, 'data_type', None)}")
    print("")

## 3. Data Extraction

Load data from a specific record set into a DataFrame for analysis.

Below we extract all records from each available record set (referenced **by `@id`**) and store them in a dictionary of DataFrames.

In [ ]:
# Extract data from each record set (by @id)
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

for rsid in record_set_ids:
    print(f"\nRecordSet @id: {rsid}")
    print(f"Fields: {dataframes[rsid].columns.tolist()}")
    display(dataframes[rsid].head())

## 4. Exploratory Data Analysis (EDA)

Apply data processing such as filtering, normalization, transformation, and grouping for the main survey data. Specify fields by their `@id`.

- We'll pick the primary survey record set (usually the largest),
- Select a numeric column (e.g., PHQ-9 Score, GAD-7 Score, or similar),
- Filter on a threshold, normalize, and group by key attribute (e.g., gender, education).

In [ ]:
# Select a representative record set, for this dataset it's typically demographics and survey
main_record_set_id = record_set_ids[0] # You may adjust this to match your main survey record set

# Print field names and choose typical numeric and categorical field @ids and names
df = dataframes[main_record_set_id]
print(f"Columns in main record set (@id={main_record_set_id}):\n{list(df.columns)}\n")

# For demonstration, we'll try PHQ-9, GAD-7 or PSQ scores -- adjust field id to match an actual field @id from print above
# Example guess: the PHQ-9 total score field might be '@id': 'phq9_score', similarly for GAD-7, PSQ, 'age', etc.
# You may need to adjust these to match the dataset!

numeric_field_id = None
group_field_id = None
# Try to autodetect commonly named fields
for col in df.columns:
    if 'phq' in col.lower():
        numeric_field_id = col
    if 'gender' in col.lower() or 'sex' in col.lower():
        group_field_id = col
if not numeric_field_id:  # fallback
    numeric_candidates = [col for col in df.columns if df[col].dtype in ['int64', 'float64']]
    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]
if not group_field_id and 'gender' in df.columns:
    group_field_id = 'gender'

print(f"Numeric field selected (for filtering/normalization): {numeric_field_id}")
print(f"Group field selected: {group_field_id}\n")

# Drop NaNs in the numeric field for fairness
filtered_df = df.dropna(subset=[numeric_field_id])

# Set a threshold to filter, e.g., PHQ-9 >= 10 (indicative of possible depression)
threshold = 10
filtered_df = filtered_df[filtered_df[numeric_field_id] >= threshold]
print(f"Filtered records with {numeric_field_id} >= {threshold}:")
display(filtered_df.head())

# Add a normalized version of the field
from sklearn.preprocessing import StandardScaler
import numpy as np
scaler = StandardScaler()
filtered_df[f"{numeric_field_id}_normalized"] = scaler.fit_transform(filtered_df[[numeric_field_id]])
print(f"Normalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by group field and show means
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped data by {group_field_id} (mean {numeric_field_id}):")
    display(grouped_df.head())

## 5. Visualization

Visualize distributions and group differences using histograms and boxplots.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# Distribution of numeric field (e.g., PHQ-9)
plt.figure(figsize=(8, 5))
sns.histplot(df[numeric_field_id], kde=True, bins=20)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# Boxplot by group field
if group_field_id in df.columns:
    plt.figure(figsize=(8, 5))
    sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion

In this notebook, we demonstrated how to load and explore the structure of the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library.

- We loaded metadata and enumerated all record sets and their fields by `@id`.
- We extracted and explored the main survey record set, filtering and normalizing a key numeric field (e.g., PHQ-9 or similar).
- We visualized distributions and group differences (e.g., by gender).

This workflow serves as a template for downstream ML experiments or statistical analysis using any dataset published in the [MLCommons Croissant](https://mlcommons.org/croissant) format.